## Pyspark vs SQL

- SQL: declarativo
- Pyspark: transformaciones encadenadas sobre DataFrames

In [0]:
%sql
SELECT * FROM samples.bakehouse.sales_customers
limit 10

In [0]:
df = spark.read.table("samples.bakehouse.sales_customers")
df.show()

In [0]:
# secondf approach
df_selected = df.select("*").limit(10)
df_selected.show()

## Distinct

In [0]:
df_distinct = df.select("country","city").distinct()
df_distinct.show()

## WHERE Filter

In [0]:
%sql
select * from samples.bakehouse.sales_customers
WHERE 1=1 -- this filter is for to write the query filter in an easier way
and country = 'Japan'
and gender = 'female'
limit 5;

In [0]:
from pyspark.sql.functions import col

df_filtered = df.where(
    (col("country") == "Japan") &
    (col("gender") == 'female')
    )
df_filtered.show()

## order by + limit

In [0]:
%sql
select * from samples.bakehouse.sales_customers
order by country
limit 10;

In [0]:
df_sorted = df.orderBy(col("first_name").asc()).limit(5)
df_sorted.show()

## Create a column

In [0]:
%sql
select 
    concat(first_name, " ", last_name) as full_name, 
    35 as age, 
    "comment" as comment,
    true as is_true
from samples.bakehouse.sales_customers

In [0]:
from pyspark.sql.functions import lit,concat,col
df_plus_columns = df.select(
    concat(col("first_name"), lit(" "), col("last_name")).alias("full_name"),
    lit(35).alias("age"),
    lit(True).alias("is_true"),
    lit("comment").alias("comment")
)
display(df_plus_columns)

## Group by + Aggregations

In [0]:
%sql
select country,count(1) as cantidad
from samples.bakehouse.sales_customers
group by country
order by cantidad

In [0]:
from pyspark.sql.functions import count
df_grouped = (
    df
    .groupBy("country")
    .agg(count("*").alias("count"))
    .orderBy(col("count").desc())
    )

df_grouped.show()

## JOIN

In [0]:
%sql
select 
    s.product,
    s.quantity,
    s.paymentMethod,
    s.dateTime,
    c.country,
    c.gender
from samples.bakehouse.sales_transactions s 
join samples.bakehouse.sales_customers c on s.customerID = c.customerID

In [0]:
df_sales_transactions = spark.read.table("samples.bakehouse.sales_transactions")

df_join = (df
           .join(df_sales_transactions, on="customerID",how='inner')
           .select("*")
           )
df_join.show()

## Cast

In [0]:
df_casted = df_sales_transactions.select(
        col("transactionID").cast("long"),
        col("customerID").cast("long"),
        col("dateTime").cast("timestamp"),
        (col("unitPrice") * col("quantity")).cast("bigint").alias("totalPrice_NEW_COLUMN")
    )
df_casted.display()

## withColumn

In [0]:
df_with_new_column = df_casted.withColumn("NewColumn", lit(366))
df_with_new_column.display()

## DROP

In [0]:
df_dropped = df_with_new_column.drop("NewColumn")
df_dropped.display()